# 🌍 ToxicGuard V4 — Çok Dilli (Multilingual) & XAI Notebook

Bu versiyonda projenizi **Kurumsal/Enterprise** seviyeye taşıyoruz.
`XLM-RoBERTa` mimarisini kullanarak hem Kaggle İngilizce veri setini hem de HuggingFace üzerinden alacağımız Türkçe Toksik veri setini birleştirip **Karma (Mixed) Veri Seti** eğiteceğiz.

---
## 💡 V4 Yenilikleri
1. **Çok Dilli Doğrudan Destek:** Çeviri servislerine (API gecikmelerine) ihtiyaç duymadan Türkçe argoyu direkt anlar.
2. **Karma Veri Seti:** 150B İngilizce metnin yanına 5B Türkçe küfür eklenir.
3. **XLM-RoBERTa:** Dünyanın en iyi çok dilli modellerinden biridir.


---
## 🔧 BÖLÜM 1 — Kurulum & Drive Bağlantısı

In [ ]:
# HÜCRE 1: Gerekli Paketlerin Kurulumu
!pip install transformers datasets evaluate accelerate lime --quiet
print("✅ Hugging Face ve LIME paketleri kuruldu!")

In [ ]:
# HÜCRE 2: Google Drive Bağlantısı
import os
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ToxicGuard'
MODELS_DIR  = os.path.join(BASE, 'models')
DATA_DIR    = os.path.join(BASE, 'data')
RESULTS_DIR = os.path.join(BASE, 'reports', 'model_results')

for d in [MODELS_DIR, DATA_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Klasör yolları ayarlandı!")

In [ ]:
# HÜCRE 3: Kütüphaneleri Dahil Etme
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    EvalPrediction
)
from sklearn.metrics import f1_score, roc_auc_score

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# GPU Aktif mi?
print(f"Cihaz: {'GPU Aktif! 🚀' if torch.cuda.is_available() else 'CPU (Yavaş) ⚠️ Runtime -> Change Runtime Type menüsünden T4 GPU seçin! '}")

---
## 📂 BÖLÜM 2 — Karma (Mixed) Veri Seti Oluşturma

In [ ]:
# HÜCRE 4: İngilizce Kaggle Verisini Yükle
TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
df_en = pd.read_csv(TRAIN_CSV)

# Hızlı prototip için veri setinden dengeli bir örneklem alınıyor
toxic = df_en[df_en[LABEL_COLS].sum(axis=1) > 0]
safe = df_en[df_en[LABEL_COLS].sum(axis=1) == 0].sample(len(toxic) * 2, random_state=42)
df_en_train = pd.concat([toxic, safe]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"İngilizce Kaggle Verisi Boyutu: {df_en_train.shape}")

In [ ]:
# HÜCRE 5: Hugging Face'ten Türkçe Toksik Verisini Çek & BİRLEŞTİR
print("🇹🇷 HuggingFace'ten Türkçe Argo/Küfür veri seti çekiliyor...")
# dataset: Overfit-GM/turkish-toxic-language
tr_dataset = load_dataset("Overfit-GM/turkish-toxic-language", split="train")

df_tr_raw = pd.DataFrame(tr_dataset)

# Türkçe veri setindeki etiketleri kendi (Kaggle) 6'lı sistemimize mapliyoruz
df_tr = pd.DataFrame()
df_tr['comment_text'] = df_tr_raw['text']
df_tr['toxic'] = df_tr_raw['is_toxic'] # 1=Toksik, 0=Değil
df_tr['severe_toxic'] = 0
df_tr['obscene'] = (df_tr_raw['target'] == 'PROFANITY').astype(int)
df_tr['threat'] = 0
df_tr['insult'] = (df_tr_raw['target'] == 'INSULT').astype(int)
df_tr['identity_hate'] = df_tr_raw['target'].isin(['RACIST', 'SEXIST']).astype(int)

print(f"Türkçe Veri Seti Boyutu: {df_tr.shape}")

# -- BİRLEŞTİRME BAŞLIYOR --
df_mixed = pd.concat([df_en_train, df_tr]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"\n🌍 KARMA (Çok Dilli) Eğitim Seti Toplam Boyutu: {df_mixed.shape}")

# HuggingFace Dataset Formatına çevirme
labels = df_mixed[LABEL_COLS].values.astype(float)
texts = df_mixed['comment_text'].tolist()

dataset = Dataset.from_dict({'text': texts, 'labels': labels})
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print("\nModel eğitimine girecek Dataset formatı:")
print(dataset)

---
## 🤖 BÖLÜM 3 — XLM-RoBERTa Model Hazırlığı

In [ ]:
# HÜCRE 6: XLM-RoBERTa Tokenizer (100+ Dil Desteği)
MODEL_NAME = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tüm dillerdeki metinler evrensel vektörlere çevriliyor (~1-2dk sürer)...")
tokenized_dataset = dataset.map(tokenize_fn, batched=True)
tokenized_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

In [ ]:
# HÜCRE 7: Çok Dilli Multi-Label Sınıflandırma Modelini Yükle
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(LABEL_COLS), 
    problem_type="multi_label_classification"
)
print("✅ XLM-RoBERTa başarıyla yüklendi.")

---
## 🔥 BÖLÜM 4 — Trainer API ile Multi-Label Eğitim

In [ ]:
# HÜCRE 8: Metrik Fonksiyonu
def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    probs = torch.sigmoid(torch.tensor(preds)).numpy()
    y_pred = (probs > 0.5).astype(int)
    y_true = p.label_ids
    
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(y_true, probs, average='macro', multi_class='ovr')
    
    return {'f1_macro': f1_macro, 'roc_auc': roc_auc}

# HÜCRE 9: Eğitim Konfigürasyonları
training_args = TrainingArguments(
    output_dir=os.path.join(MODELS_DIR, 'xlm_roberta_v4_checkpoints'),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True, # GPU verimliliği ve hızı için (Mixed Precision)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    compute_metrics=compute_metrics
)

In [ ]:
# HÜCRE 10: Eğitimi Başlat!
print("🚀 XLM-RoBERTa Eğitimi Başlıyor! (T4 GPU ile ~10-20 dk sürer)")
trainer.train()
print("✅ Türkçe+İngilizce Çok Dilli Model Eğitimi Tamamlandı!")

---
## 💾 BÖLÜM 5 — Altın Standart V4'ü Kaydet

In [ ]:
# HÜCRE 11: V4 Modeli Kaydetme
v4_model_path = os.path.join(MODELS_DIR, 'toxicguard_v4_multilingual')

trainer.save_model(v4_model_path)
tokenizer.save_pretrained(v4_model_path)

print(f"🎉 HARİKA! State-of-the-Art Çok Dilli (Türkçe destekli) V4 Modeliniz şuraya kaydedildi:\n{v4_model_path}")
print("\nArtık bu klasörü indirip arayüze LIME (XAI) ekleyerek yerel server'da kurumsal düzeyde çalışabiliriz!")